## 22127260 - Bùi Công Mậu
## 22127400 - Thái Hữu Thọ

In [1]:
import pandas as pd
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import train_test_split
import lightgbm as lgb 
import time 
import os
import psutil 
import numpy as np

In [ ]:
# Đọc dữ liệu
train_data = pd.read_csv('../Data/train_normalized.csv')
test_data = pd.read_csv('../Data/test_normalized.csv')

# Tách đặc trưng và biến mục tiêutiêu
X = train_data.drop(columns=["id", "target"])
Y = train_data["target"]
X_test = test_data.drop(columns=["id"])

# Chuyển đổi loglog
Y_log = np.log1p(Y)

# Chia tập train/ validationon
X_train, X_val, Y_train_log, Y_val_log = train_test_split(X, Y_log, test_size=0.2, random_state=42)

# Đo thời gian
start_time = time.time()
process = psutil.Process(os.getpid())

# Chuẩn bị dữ liệu LightGBM 
lgb_train = lgb.Dataset(X_train, Y_train_log)
lgb_val = lgb.Dataset(X_val, Y_val_log, reference=lgb_train)

# parameters cho LightGBM 
params = {
    'objective': 'regression',
    'metric': 'rmse',  
    'boosting_type': 'gbdt',
    'learning_rate': 0.001, 
    'num_leaves': 64,
    'max_depth': 8,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'lambda_l1': 1.5,  
    'lambda_l2': 1.5,  
    'verbose': -1,  
    'random_state': 42
}

# Huấn luyện mô hình với early stopping 
model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'valid'],
    num_boost_round=10000, 
    callbacks=[
        lgb.early_stopping(stopping_rounds=200), 
        lgb.log_evaluation(period=100)  
    ]
)

# Dự doán trên tập validation 
Y_pred_val_log = model.predict(X_val, num_iteration=model.best_iteration)
Y_pred_val = np.expm1(Y_pred_val_log)  
Y_val = np.expm1(Y_val_log) 

# Dự đoán trên tập test 
Y_test_pred_log = model.predict(X_test, num_iteration=model.best_iteration)
Y_test_pred = np.expm1(Y_test_pred_log) 

# Đo thời gian và bộ nhớ 
end_time = time.time()
training_time = end_time - start_time
memory_usage = process.memory_info().rss / (1024 ** 2)

print(f"Thời gian huấn luyện: {training_time:.2f} giây")
print(f"Mức sử dụng bộ nhớ: {memory_usage:.2f} MB")

# Tính RMSLE trên tập validation 
rmsle = root_mean_squared_log_error(Y_val, np.clip(Y_pred_val, 0, None))
print("RMSLE trên tập validation:", rmsle)

# Xuất dữ liệu 
submission = pd.DataFrame({"id": test_data["id"], "target": Y_test_pred})
submission.to_csv("../Data/Phase 3.1.csv", index=False)


Training until validation scores don't improve for 200 rounds
[100]	train's rmse: 1.0916	valid's rmse: 1.09239
[200]	train's rmse: 1.08772	valid's rmse: 1.08856
[300]	train's rmse: 1.08423	valid's rmse: 1.08513
[400]	train's rmse: 1.08123	valid's rmse: 1.08218
[500]	train's rmse: 1.07818	valid's rmse: 1.07917
[600]	train's rmse: 1.07547	valid's rmse: 1.07652
[700]	train's rmse: 1.07308	valid's rmse: 1.07417
[800]	train's rmse: 1.0708	valid's rmse: 1.07194
[900]	train's rmse: 1.0689	valid's rmse: 1.07009
[1000]	train's rmse: 1.06711	valid's rmse: 1.06834
[1100]	train's rmse: 1.0657	valid's rmse: 1.06699
[1200]	train's rmse: 1.0643	valid's rmse: 1.06563
[1300]	train's rmse: 1.06315	valid's rmse: 1.06453
[1400]	train's rmse: 1.06195	valid's rmse: 1.06338
[1500]	train's rmse: 1.0608	valid's rmse: 1.06228
[1600]	train's rmse: 1.05976	valid's rmse: 1.0613
[1700]	train's rmse: 1.05894	valid's rmse: 1.06052
[1800]	train's rmse: 1.05814	valid's rmse: 1.05977
[1900]	train's rmse: 1.05741	valid's